# Kinesin Motor Proteins — Phylogenetic Analysis

---

## Overview

This notebook performs a **complete phylogenetic analysis** of the Kinesin motor protein superfamily.

### Pipeline
1. Sequence retrieval from NCBI via Biopython
2. Multiple sequence alignment with MUSCLE
3. Maximum likelihood tree inference with RAxML / FastTree
4. Tree visualization with Matplotlib and ETE3
5. Interactive visualization with Auspice (Nextstrain)



## 1. Install Dependencies

In [ ]:
import subprocess, sys

# Python packages
pkgs = ['biopython', 'ete3', 'matplotlib', 'seaborn', 'numpy', 'pandas', 'plotly', 'scipy']
for p in pkgs:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', p])
    print(f'  {p}')

# System alignment / tree tools
!apt-get install -qq -y muscle raxml mafft fasttree 2>/dev/null | tail -1

print('\n Dependencies ready.')

## 2. Imports & Configuration

In [ ]:
import os, re, json, warnings, time
from io import StringIO
from pathlib import Path
warnings.filterwarnings('ignore')

from Bio import Entrez, SeqIO, AlignIO, Phylo
from Bio.Phylo.TreeConstruction import DistanceCalculator, DistanceTreeConstructor
from Bio.Align import MultipleSeqAlignment

import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import matplotlib.gridspec as gridspec
from matplotlib.lines import Line2D
import seaborn as sns
import numpy as np
import pandas as pd

# ── NCBI Entrez setup ──────────────────────────────────────────────
Entrez.email = 'user@example.com'   # Required by NCBI — replace with a valid address

# ── Plot defaults ──────────────────────────────────────────────────
plt.rcParams.update({
    'figure.dpi': 150,
    'font.family': 'DejaVu Sans',
    'axes.spines.top': False,
    'axes.spines.right': False,
})

# ── Output directories ─────────────────────────────────────────────
for d in ['data/sequences', 'data/alignments', 'figures', 'results', 'auspice']:
    Path(d).mkdir(parents=True, exist_ok=True)

print('Setup complete')
print(f'Working directory: {Path.cwd()}')

## 3. Define Kinesin Families & Accessions

In [ ]:
# ── Kinesin family definitions ─────────────────────────────────────
# Each entry: [AccessionID, Family, Species_abbrev, GeneName]
# Source: Duke Kinesin Site (https://sites.duke.edu/kinesin/kinesin-tree/)

KINESIN_ACCESSIONS = [
    # Kinesin-1 (KHC) — Plus-end, organelle transport
    ('NP_004983', 'Kinesin-1',  'Hs', 'KIF5A'),
    ('NP_004521', 'Kinesin-1',  'Hs', 'KIF5B'),
    ('NP_004954', 'Kinesin-1',  'Hs', 'KIF5C'),
    ('NP_524153', 'Kinesin-1',  'Dm', 'KHC'),
    ('NP_499043', 'Kinesin-1',  'Ce', 'UNC-116'),

    # Kinesin-2 (KRP85/95) — Plus-end, IFT/cilia
    ('NP_009376', 'Kinesin-2',  'Hs', 'KIF3A'),
    ('NP_000219', 'Kinesin-2',  'Hs', 'KIF3B'),
    ('NP_524582', 'Kinesin-2',  'Dm', 'KLP64D'),

    # Kinesin-3 (Unc104) — Plus-end, synaptic vesicle
    ('NP_004270', 'Kinesin-3',  'Hs', 'KIF1A'),
    ('NP_055046', 'Kinesin-3',  'Hs', 'KIF1B'),
    ('NP_493636', 'Kinesin-3',  'Ce', 'UNC-104'),

    # Kinesin-4 (ChrKin) — Plus-end, chromosome
    ('NP_002253', 'Kinesin-4',  'Hs', 'KIF4A'),
    ('NP_524914', 'Kinesin-4',  'Dm', 'KLP3A'),

    # Kinesin-5 (BimC/Eg5) — Plus-end, bipolar spindle
    ('NP_004514', 'Kinesin-5',  'Hs', 'KIF11'),
    ('NP_524907', 'Kinesin-5',  'Dm', 'KLP61F'),
    ('NP_013710', 'Kinesin-5',  'Sc', 'CIN8'),

    # Kinesin-6 (MKLP1) — Plus-end, cytokinesis
    ('NP_006255', 'Kinesin-6',  'Hs', 'KIF23'),
    ('NP_492484', 'Kinesin-6',  'Ce', 'ZEN-4'),

    # Kinesin-7 (CENP-E) — Plus-end, kinetochore
    ('NP_001800', 'Kinesin-7',  'Hs', 'KIF10'),

    # Kinesin-8 (Kip3) — Plus-end, MT depolymerization
    ('NP_055732', 'Kinesin-8',  'Hs', 'KIF18A'),
    ('NP_013490', 'Kinesin-8',  'Sc', 'KIP3'),

    # Kinesin-13 (MCAK) — Depolymerizer
    ('NP_006058', 'Kinesin-13', 'Hs', 'KIF2A'),
    ('NP_006836', 'Kinesin-13', 'Hs', 'KIF2C'),

    # Kinesin-14 (C-Terminal Motor) — MINUS-END directed
    ('NP_477177', 'Kinesin-14', 'Dm', 'NCD'),
    ('NP_013491', 'Kinesin-14', 'Sc', 'KAR3'),
    ('NP_003147', 'Kinesin-14', 'Hs', 'KIFC1'),
    ('NP_055854', 'Kinesin-14', 'Hs', 'KIFC2'),
    ('NP_009118', 'Kinesin-14', 'Hs', 'KIFC3'),
]

# ── Family colour palette ──────────────────────────────────────────
FAMILY_COLORS = {
    'Kinesin-1':  '#2196F3',
    'Kinesin-2':  '#4CAF50',
    'Kinesin-3':  '#FF9800',
    'Kinesin-4':  '#9C27B0',
    'Kinesin-5':  '#00BCD4',
    'Kinesin-6':  '#FF5722',
    'Kinesin-7':  '#607D8B',
    'Kinesin-8':  '#8BC34A',
    'Kinesin-13': '#FFC107',
    'Kinesin-14': '#F44336',   # Red — minus-end
}

df_acc = pd.DataFrame(KINESIN_ACCESSIONS, columns=['Accession','Family','Species','Gene'])
print(f'📋 {len(df_acc)} sequences defined across {df_acc.Family.nunique()} families')
display(df_acc.groupby('Family').size().rename('count').reset_index())

## 4. Fetch Sequences from NCBI

In [ ]:
def fetch_sequences(accessions_df, output_fasta, delay=0.4):
    """
    Fetch protein sequences from NCBI and write to FASTA.
    Labels: >{Species}_{Gene}_{Family}  (e.g. >Hs_KIF5A_Kinesin-1)
    """
    records = []
    failed  = []

    for _, row in accessions_df.iterrows():
        acc    = row['Accession']
        label  = f"{row['Species']}_{row['Gene']}_{row['Family'].replace('-','')}"
        try:
            handle = Entrez.efetch(db='protein', id=acc, rettype='fasta', retmode='text')
            rec    = SeqIO.read(handle, 'fasta')
            handle.close()
            rec.id          = label
            rec.description = ''
            records.append(rec)
            print(f'  ✅  {acc:15s}  {label}')
        except Exception as e:
            failed.append(acc)
            print(f'  ⚠️   {acc:15s}  FAILED — {e}')
        time.sleep(delay)   # Respect NCBI rate limit (3 req/s)

    SeqIO.write(records, output_fasta, 'fasta')
    print(f'\n💾 {len(records)} sequences saved → {output_fasta}')
    if failed:
        print(f'   Failed accessions: {failed}')
    return records

fasta_path = 'data/sequences/kinesin_all_families.fasta'
records = fetch_sequences(df_acc, fasta_path)

print(f'\nSequence length range: {min(len(r.seq) for r in records)} – {max(len(r.seq) for r in records)} aa')

## 5. Sequence Quality Control & Statistics

In [ ]:
# Build sequence stats dataframe
stats = []
for rec in records:
    parts  = rec.id.split('_')
    family = '_'.join(parts[2:]).replace('Kinesin','Kinesin-')
    seq    = str(rec.seq)
    stats.append({
        'ID':     rec.id,
        'Family': family,
        'Length': len(seq),
        'GC_pct': 0,   # protein; placeholder
        'pct_charged': sum(seq.count(aa) for aa in 'DEKR') / len(seq) * 100,
        'pct_hydrophobic': sum(seq.count(aa) for aa in 'VILMFYW') / len(seq) * 100,
    })

df_stats = pd.DataFrame(stats)
display(df_stats.groupby('Family')[['Length','pct_charged','pct_hydrophobic']].mean().round(1))

# ── Figure 1: Sequence length distribution ─────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('Kinesin Superfamily — Sequence Statistics', fontsize=14, fontweight='bold')

# Panel A — violin plot per family
ax = axes[0]
families = df_stats['Family'].unique()
data_grouped = [df_stats[df_stats['Family'] == f]['Length'].values for f in families]
vp = ax.violinplot(data_grouped, showmedians=True)
for i, (patch, fam) in enumerate(zip(vp['bodies'], families)):
    col = FAMILY_COLORS.get(fam, '#888')
    patch.set_facecolor(col)
    patch.set_alpha(0.7)
ax.set_xticks(range(1, len(families)+1))
ax.set_xticklabels(families, rotation=45, ha='right', fontsize=8)
ax.set_ylabel('Sequence Length (aa)')
ax.set_title('A — Length Distribution per Family')
ax.grid(axis='y', alpha=0.3)

# Panel B — amino acid composition bars
ax2 = axes[1]
comp_df = df_stats.groupby('Family')[['pct_charged','pct_hydrophobic']].mean()
x = np.arange(len(comp_df))
w = 0.35
ax2.bar(x - w/2, comp_df['pct_charged'],    width=w, label='Charged (D/E/K/R)', color='#5C85D6', alpha=0.85)
ax2.bar(x + w/2, comp_df['pct_hydrophobic'], width=w, label='Hydrophobic',       color='#E07B54', alpha=0.85)
ax2.set_xticks(x)
ax2.set_xticklabels(comp_df.index, rotation=45, ha='right', fontsize=8)
ax2.set_ylabel('Amino Acid Frequency (%)')
ax2.set_title('B — AA Composition per Family')
ax2.legend(fontsize=8)
ax2.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.savefig('figures/sequence_length_distribution.png', dpi=150, bbox_inches='tight')
plt.show()
print(' figures/sequence_length_distribution.png')

## 6. Multiple Sequence Alignment (MUSCLE)

In [ ]:
import shutil

IN_FASTA  = 'data/sequences/kinesin_all_families.fasta'
ALN_FASTA = 'data/alignments/kinesin_aligned.fasta'

def run_muscle(input_fasta, output_fasta):
    """Run MUSCLE alignment; fall back to Biopython NW pairwise if MUSCLE absent."""
    if shutil.which('muscle'):
        cmd = f'muscle -align {input_fasta} -output {output_fasta}'
        result = subprocess.run(cmd, shell=True, capture_output=True, text=True)
        if result.returncode != 0:
            raise RuntimeError(result.stderr)
        method = 'MUSCLE v5'
    elif shutil.which('mafft'):
        cmd = f'mafft --auto {input_fasta} > {output_fasta}'
        subprocess.run(cmd, shell=True, check=True)
        method = 'MAFFT'
    else:
        # Pure Biopython fallback — pairwise then write
        from Bio import pairwise2
        recs = list(SeqIO.parse(input_fasta, 'fasta'))
        SeqIO.write(recs, output_fasta, 'fasta')
        method = 'Biopython (no alignment; install MUSCLE for production)'
    return method

method = run_muscle(IN_FASTA, ALN_FASTA)
alignment = AlignIO.read(ALN_FASTA, 'fasta')

print(f'   Alignment method : {method}')
print(f'   Sequences        : {len(alignment)}')
print(f'   Alignment length : {alignment.get_alignment_length()} columns')

In [ ]:
# ── Visualise alignment (first 100 columns) ────────────────────────
n_seq  = len(alignment)
n_cols = min(100, alignment.get_alignment_length())

AA_COLOR = {
    'A':'#80B1D3','V':'#80B1D3','I':'#80B1D3','L':'#80B1D3','M':'#80B1D3',
    'F':'#FB8072','W':'#FB8072','Y':'#FB8072',
    'K':'#FDB462','R':'#FDB462','H':'#FDB462',
    'D':'#FCCDE5','E':'#FCCDE5',
    'S':'#B3DE69','T':'#B3DE69',
    'N':'#FFFFB3','Q':'#FFFFB3',
    'C':'#BC80BD','G':'#CCEBC5','P':'#FFED6F',
    '-':'#EEEEEE',
}

fig, ax = plt.subplots(figsize=(18, max(4, n_seq * 0.3)))
for i, rec in enumerate(alignment):
    seq_slice = str(rec.seq)[:n_cols]
    for j, aa in enumerate(seq_slice):
        color = AA_COLOR.get(aa, '#FFFFFF')
        ax.add_patch(mpatches.Rectangle((j, n_seq - i - 1), 1, 1,
                                         color=color, lw=0))
        if n_cols <= 60:
            ax.text(j + 0.5, n_seq - i - 0.5, aa, ha='center', va='center',
                    fontsize=6, fontweight='bold')

ax.set_xlim(0, n_cols)
ax.set_ylim(0, n_seq)
ax.set_yticks(np.arange(n_seq) + 0.5)
ax.set_yticklabels([r.id for r in alignment][::-1], fontsize=7)
ax.set_xlabel(f'Alignment Position (first {n_cols} columns shown)')
ax.set_title('Multiple Sequence Alignment — Kinesin Superfamily', fontweight='bold')

# Legend
legend_items = [
    mpatches.Patch(color='#80B1D3', label='Hydrophobic'),
    mpatches.Patch(color='#FB8072', label='Aromatic'),
    mpatches.Patch(color='#FDB462', label='Positive'),
    mpatches.Patch(color='#FCCDE5', label='Negative'),
    mpatches.Patch(color='#B3DE69', label='Polar'),
    mpatches.Patch(color='#EEEEEE', label='Gap'),
]
ax.legend(handles=legend_items, loc='upper right', fontsize=7, ncol=3,
          bbox_to_anchor=(1.0, -0.07), frameon=False)
plt.tight_layout()
plt.savefig('figures/alignment_view.png', dpi=150, bbox_inches='tight')
plt.show()
print('figures/alignment_view.png')

## 7. Phylogenetic Tree Inference

In [ ]:
# ── Option A: RAxML (Maximum Likelihood) ──────────────────────────
def run_raxml(alignment_file, out_prefix, model='PROTGAMMALG', bootstraps=100):
    """Run RAxML with bootstrap support."""
    if not shutil.which('raxmlHPC') and not shutil.which('raxml'):
        return None, 'raxml_not_found'

    raxml_bin = 'raxmlHPC' if shutil.which('raxmlHPC') else 'raxml'

    # Remove old run files
    for f in Path('.').glob(f'RAxML_*{out_prefix}*'):
        f.unlink()

    cmd = (
        f'{raxml_bin} -f a '
        f'-m {model} '
        f'-p 42 -x 42 '
        f'-# {bootstraps} '
        f'-s {alignment_file} '
        f'-n {out_prefix} '
        f'-w {Path("results/raxml_kinesin").resolve()}'
    )
    result = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    tree_file = f'results/raxml_kinesin/RAxML_bipartitions.{out_prefix}'
    return tree_file, result.returncode

# ── Option B: FastTree (fallback) ─────────────────────────────────
def run_fasttree(alignment_file, tree_file):
    """Run FastTree — fast approximate ML."""
    cmd = f'FastTree -lg -gamma -quiet {alignment_file} > {tree_file}'
    r = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    return tree_file, r.returncode

# ── Option C: Biopython NJ (pure-Python fallback) ─────────────────
def run_nj_tree(alignment_file, tree_file):
    """Neighbour-Joining tree — no external tools required."""
    aln  = AlignIO.read(alignment_file, 'fasta')
    calc = DistanceCalculator('blosum62')
    dm   = calc.get_distance(aln)
    ctor = DistanceTreeConstructor(calc, method='nj')
    tree = ctor.build_tree(aln)
    Phylo.write(tree, tree_file, 'newick')
    return tree_file, 0

# ── Run best available method ─────────────────────────────────────
TREE_FILE = 'results/kinesin_tree.nwk'

if shutil.which('raxmlHPC') or shutil.which('raxml'):
    tree_file, code = run_raxml('data/alignments/kinesin_aligned.fasta', 'kinesin')
    if code == 0:
        import shutil as sh
        sh.copy(tree_file, TREE_FILE)
        print('✅ RAxML tree complete')
    else:
        print('⚠️  RAxML failed — falling back to FastTree')
        run_fasttree('data/alignments/kinesin_aligned.fasta', TREE_FILE)
elif shutil.which('FastTree') or shutil.which('fasttree'):
    run_fasttree('data/alignments/kinesin_aligned.fasta', TREE_FILE)
    print('✅ FastTree complete')
else:
    run_nj_tree('data/alignments/kinesin_aligned.fasta', TREE_FILE)
    print('✅ NJ tree (Biopython) complete — install RAxML/FastTree for ML trees')

tree = Phylo.read(TREE_FILE, 'newick')
print(f'\nTree has {tree.count_terminals()} leaves and {len(tree.get_nonterminals())} internal nodes')

## 8. Tree Visualization

In [ ]:
def label_to_family(label):
    """Extract kinesin family from sequence label."""
    # Label format: Species_Gene_KinesinXX
    parts = label.split('_')
    for p in parts:
        m = re.match(r'Kinesin(\d+)', p)
        if m:
            return f'Kinesin-{m.group(1)}'
    return 'Unknown'

# Colour each terminal clade
def color_tree(tree, color_map):
    for clade in tree.find_clades():
        if clade.is_terminal():
            fam = label_to_family(clade.name or '')
            clade.color = color_map.get(fam, '#888888')
        else:
            clade.color = '#555555'

color_tree(tree, FAMILY_COLORS)

# ── Rectangular tree ──────────────────────────────────────────────
n_tips  = tree.count_terminals()
fig_h   = max(8, n_tips * 0.35)
fig, ax = plt.subplots(figsize=(14, fig_h))

Phylo.draw(tree, axes=ax, do_show=False,
           label_func=lambda c: c.name if c.is_terminal() else '',
           branch_labels=lambda c: (f'{c.confidence:.0f}' if c.confidence and not c.is_terminal() else ''),
           label_colors=lambda c: FAMILY_COLORS.get(label_to_family(c.name or ''), '#333'))

ax.set_title('Kinesin Motor Protein Superfamily — Phylogenetic Tree', fontsize=13, fontweight='bold', pad=12)
ax.set_xlabel('Branch Length (substitutions per site)')
ax.spines[['top','right']].set_visible(False)

# Legend
handles = [mpatches.Patch(color=c, label=f) for f, c in FAMILY_COLORS.items()]
ax.legend(handles=handles, title='Kinesin Family', title_fontsize=9,
          fontsize=8, loc='lower right', framealpha=0.9, ncol=2)

plt.tight_layout()
plt.savefig('figures/kinesin_phylotree.png', dpi=150, bbox_inches='tight')
plt.show()
print('💾 figures/kinesin_phylotree.png')

In [ ]:
# ── Circular / radial tree ─────────────────────────────────────────
# Implemented with polar matplotlib axes

def draw_circular_tree(tree, color_map, output_file):
    terminals = tree.get_terminals()
    n = len(terminals)
    angles = {t.name: 2 * np.pi * i / n for i, t in enumerate(terminals)}

    fig = plt.figure(figsize=(12, 12))
    ax  = fig.add_subplot(111, projection='polar')
    ax.set_theta_zero_location('N')
    ax.set_theta_direction(-1)
    ax.axis('off')
    ax.set_title('Kinesin Superfamily — Circular Phylogeny', fontsize=13,
                 fontweight='bold', pad=20)

    # Draw leaf labels
    for t in terminals:
        ang = angles[t.name]
        fam = label_to_family(t.name or '')
        col = color_map.get(fam, '#888')
        ax.text(ang, 1.05, t.name, ha='center', va='center',
                fontsize=6, color=col, rotation=np.degrees(ang),
                rotation_mode='anchor')
        ax.plot([ang, ang], [0.95, 1.0], color=col, lw=2)

    # Draw coloured arcs per family
    for fam, col in color_map.items():
        fam_tips = [t for t in terminals if label_to_family(t.name) == fam]
        if len(fam_tips) < 2:
            continue
        fam_angles = [angles[t.name] for t in fam_tips]
        a_start = min(fam_angles)
        a_end   = max(fam_angles)
        arc_a   = np.linspace(a_start, a_end, 50)
        ax.plot(arc_a, [0.92] * 50, color=col, lw=3, alpha=0.8, solid_capstyle='round')

    # Legend
    handles = [mpatches.Patch(color=c, label=f) for f, c in color_map.items()]
    ax.legend(handles=handles, title='Kinesin Family', fontsize=7,
              loc='lower center', bbox_to_anchor=(0.5, -0.05), ncol=5, frameon=False)

    plt.savefig(output_file, dpi=150, bbox_inches='tight')
    plt.show()
    print(f'💾 {output_file}')

draw_circular_tree(tree, FAMILY_COLORS, 'figures/kinesin_circular_tree.png')

## 9. Auspice (Nextstrain) Interactive Export

In [ ]:
def build_auspice_json(tree_file, accessions_df, output_json):
    """
    Build a minimal Auspice v2 JSON for interactive viewing at
    https://auspice.us  — drag-and-drop the output JSON.
    """

    def clade_to_dict(clade, depth=0):
        label = (clade.name or f'node_{depth}_{id(clade)}')
        fam   = label_to_family(label)
        row   = accessions_df[accessions_df.apply(
                    lambda r: f"{r['Species']}_{r['Gene']}" in label, axis=1)]
        species = row['Species'].values[0] if len(row) else 'Unknown'

        node = {
            'name': label,
            'node_attrs': {
                'div':    clade.branch_length or 0,
                'family': {'value': fam},
                'species': {'value': species},
                'directionality': {
                    'value': 'Minus-end' if fam == 'Kinesin-14' else 'Plus-end'
                },
            }
        }
        if clade.confidence is not None:
            node['node_attrs']['bootstrap'] = {'value': float(clade.confidence)}

        if clade.clades:
            node['children'] = [clade_to_dict(c, depth+1) for c in clade.clades]
        return node

    tree_obj = Phylo.read(tree_file, 'newick')
    root_dict = clade_to_dict(tree_obj.root)

    auspice = {
        'version': 'v2',
        'meta': {
            'title': 'Kinesin Motor Protein Superfamily Phylogeny',
            'description': (
                'Maximum-likelihood phylogeny of kinesin motor proteins (Kinesin-1 to '
                'Kinesin-14). Built with Biopython + RAxML. '
                'Reference: Duke Kinesin Site (https://sites.duke.edu/kinesin/).'
            ),
            'colorings': [
                {'key': 'family',         'title': 'Kinesin Family',   'type': 'categorical'},
                {'key': 'species',        'title': 'Species',          'type': 'categorical'},
                {'key': 'directionality', 'title': 'Motor Direction',  'type': 'categorical'},
            ],
            'display_defaults': {'color_by': 'family'},
            'panels': ['tree'],
        },
        'tree': root_dict,
    }

    with open(output_json, 'w') as fh:
        json.dump(auspice, fh, indent=2)

    print(f'✅ Auspice JSON written → {output_json}')
    print('   View interactively: drag the file to https://auspice.us')

build_auspice_json(TREE_FILE, df_acc, 'auspice/kinesin_auspice.json')

## 10.  Bootstrap Support Summary

In [ ]:
tree_bs = Phylo.read(TREE_FILE, 'newick')
bs_values = [c.confidence for c in tree_bs.get_nonterminals()
             if c.confidence is not None]

if bs_values:
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    fig.suptitle('Bootstrap Support — Kinesin Superfamily Tree', fontweight='bold')

    # Histogram
    ax1 = axes[0]
    ax1.hist(bs_values, bins=20, color='#4C72B0', edgecolor='white', alpha=0.85)
    ax1.axvline(70,  color='#DD4444', lw=2, ls='--', label='70% threshold')
    ax1.axvline(np.mean(bs_values), color='#22AA22', lw=2, ls='-', label=f'Mean = {np.mean(bs_values):.1f}')
    ax1.set_xlabel('Bootstrap Support (%)')
    ax1.set_ylabel('Number of Nodes')
    ax1.set_title('A — Distribution of Bootstrap Values')
    ax1.legend(fontsize=8)
    ax1.grid(alpha=0.3)

    # Cumulative
    ax2 = axes[1]
    sorted_bs = np.sort(bs_values)
    cdf = np.arange(1, len(sorted_bs)+1) / len(sorted_bs)
    ax2.plot(sorted_bs, cdf, color='#4C72B0', lw=2)
    ax2.axvline(70, color='#DD4444', lw=2, ls='--', label='70% threshold')
    ax2.fill_between(sorted_bs, cdf, alpha=0.15, color='#4C72B0')
    ax2.set_xlabel('Bootstrap Support (%)')
    ax2.set_ylabel('Cumulative Fraction')
    ax2.set_title('B — Cumulative Bootstrap Distribution')
    ax2.legend(fontsize=8)
    ax2.grid(alpha=0.3)

    plt.tight_layout()
    plt.savefig('figures/bootstrap_support_heatmap.png', dpi=150, bbox_inches='tight')
    plt.show()

    well_supported = sum(1 for v in bs_values if v >= 70)
    print(f'Bootstrap summary:')
    print(f'  Total internal nodes : {len(bs_values)}')
    print(f'  Mean bootstrap       : {np.mean(bs_values):.1f}%')
    print(f'  Well-supported (≥70) : {well_supported}/{len(bs_values)} ({100*well_supported/len(bs_values):.0f}%)')
else:
    print('ℹ️  No bootstrap values in tree (NJ tree used). Run RAxML for bootstrap support.')
    print('💾 figures/bootstrap_support_heatmap.png')

## Summary

| Output | File |
|--------|------|
| Raw sequences | `data/sequences/kinesin_all_families.fasta` |
| Alignment | `data/alignments/kinesin_aligned.fasta` |
| Newick tree | `results/kinesin_tree.nwk` |
| Rectangular tree figure | `figures/kinesin_phylotree.png` |
| Circular tree figure | `figures/kinesin_circular_tree.png` |
| Sequence stats figure | `figures/sequence_length_distribution.png` |
| Bootstrap figure | `figures/bootstrap_support_heatmap.png` |
| Auspice JSON | `auspice/kinesin_auspice.json` |

**Interactive viewing**: drag `auspice/kinesin_auspice.json` to [https://auspice.us](https://auspice.us)

---
